In [0]:
from pyspark.sql import functions as F

# Unity Catalog Delta table monitored by the Lakeflow Job's table-update trigger
landing_table = "predictivemaintenancecatalog.raw_data.landing_testdata"

# Storage volume for the streaming checkpoint (not the source of incoming rows)
landing_path = "/Volumes/predictivemaintenancecatalog/raw_data/landing"

# New checkpoint: do NOT reuse the previous volume-based streaming checkpoint
# Keep this stable across future job runs; do not delete it after initial setup.
data_checkpoint_location = f"{landing_path}/_checkpoints/streaming_data_uc_landing_v1"

# Bronze destination (unchanged)
bronze_table = (
    "predictivemaintenancecatalog."
    "bronze_data_not_quality_checked.streaming"
)

# Read commits from the registered Delta table, NOT the volume path.
# Version 0 contains the initial 2010 rows already handled separately.
# IMPORTANT: the first run must happen AFTER the table has a version 1 commit.
# 'startingVersion' is used only when this new checkpoint initializes.
raw_data_streaming = (
    spark.readStream
    .format("delta")
    .option("startingVersion", 1)
    .table(landing_table)
)

# Add ingestion metadata
data_bronze = (
    raw_data_streaming
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source", F.lit("delta_landing"))
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# Process available new commits, then exit so the job can finish.
query = (
    data_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", data_checkpoint_location)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

